# Lab 4. Naive Bayes

## I. Phân loại Naive Bayes sử dụng thư viện sklearn trên dataset nhỏ 

### Bài toán dự đoán rủi ro tín dụng
- Cho danh sách những người vay tiền với các đặc trưng được quan sát bao gồm: Người vay có sở hữu nhà ở hay không (Home Owner); Tình trạng hôn nhân (Marital Status); Thu nhập hàng năm (Annual Income); và Nhận định người vay có bị vỡ nợ hay không (Defaulted Borrower). Trong đó Defaulted Borrower là nhãn lớp cho biết người vay có trả được khoản tiền đã vay hay không? Dữ liệu được lưu trong file credit_data.txt.
- Yêu cầu: Sử dụng thuật toán Naïve Bayes để dự đoán xác suất một người có vỡ nợ hay không dựa vào các đặc trưng Home Owner, Marital Status và Annual Income.

### 1. Import các thư viện cần thiết

In [2]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB, ComplementNB

### 2. Cấu hình dữ liệu

In [3]:
data = pd.DataFrame([
    ["Yes", "Single", "High", "No"],
    ["No", "Married", "High", "No"],
    ["No", "Single", "Low", "No"],
    ["Yes", "Married", "High", "No"],
    ["No", "Divorced", "Low", "Yes"],
    ["No", "Married", "Low", "No"],
    ["Yes", "Divorced", "High", "No"],
    ["No", "Single", "Low", "Yes"],
    ["No", "Married", "Low", "No"],
    ["No", "Single", "Low", "Yes"]
], columns=["Home Owner", "Marital Status", "Annual Income", "Defaulted Borrower"])

In [ ]:
# Yêu cầu 1: Chuẩn bị dữ liệu cho mô hình học máy (X: lưu giá trị của các cột đặc trưng; y: lưu giá trị cột nhãn)
######
# Code

X = data[['Home Owner', 'Marital Status', 'Annual Income']]
#X = data.drop("Defaulted Borrower", axis=1)
y = data['Defaulted Borrower']

print('Dữ liệu đặc trưng X:')
print(X)
print('\nNhãn y:')
print(y)
######

In [5]:
# Yêu cầu 2: Chuẩn hóa dữ liệu từ dạng phân loại sang dạng số học.
# Gợi ý: Có thể sử dụng các kỹ thuật Label Encoding, One-Hot Encoding,...
######
# Code

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from scipy.sparse import issparse

# One-Hot Encoding cho X (dùng cho Gaussian, Multinomial, Bernoulli, Complement NB)
ct = ColumnTransformer([
    ('encoder', OneHotEncoder(), ['Home Owner', 'Marital Status', 'Annual Income'])
])
X_ohe = ct.fit_transform(X)
if issparse(X_ohe):
    X_ohe = X_ohe.toarray()

# Ordinal Encoding cho X (dùng riêng cho Categorical NB)
oe = OrdinalEncoder()
X_ordinal = oe.fit_transform(X).astype(int)

# Label Encoding cho y
le = LabelEncoder()
y_enc = le.fit_transform(y)

print('X sau One-Hot Encoding (dùng cho GaussianNB, MultinomialNB, BernoulliNB, ComplementNB):')
print(X_ohe)
print('\nX sau Ordinal Encoding (dùng cho CategoricalNB):')
print(X_ordinal)
print('\ny sau Label Encoding:', y_enc, '| Classes:', le.classes_)
######

### 3. Training và Testing mô hình Naive Bayes với các phân phối xác suất khác nhau

In [6]:
# Yêu cầu 3: Chuẩn bị một mẫu dữ liệu mới và chuẩn hóa về giá trị số để kiểm tra kết quả phân lớp của mô hình
######
# Code

# Tạo mẫu mới: No (Home Owner), Married (Marital Status), Low (Annual Income)
new_sample = pd.DataFrame([['No', 'Married', 'Low']],
                           columns=['Home Owner', 'Marital Status', 'Annual Income'])

# Mã hóa One-Hot (cho GaussianNB, MultinomialNB, BernoulliNB, ComplementNB)
new_ohe = ct.transform(new_sample)
if issparse(new_ohe):
    new_ohe = new_ohe.toarray()

# Mã hóa Ordinal (cho CategoricalNB)
new_ordinal = oe.transform(new_sample).astype(int)

print('Mẫu mới (One-Hot):', new_ohe)
print('Mẫu mới (Ordinal):', new_ordinal)
######

In [7]:
# Yêu cầu 4: Huấn luyện và kiểm tra mô hình Gaussian Naive Bayes (sử dụng phân phối chuẩn Gauss)
######
# Code

gnb = GaussianNB()
gnb.fit(X_ohe, y_enc)

pred_gnb = gnb.predict(new_ohe)
prob_gnb = gnb.predict_proba(new_ohe)

print('=== Gaussian Naive Bayes ===')
print(f'Kết quả dự đoán: {le.inverse_transform(pred_gnb)[0]}')
print(f'Xác suất [No, Yes]: {prob_gnb.round(4)}')
######

In [8]:
# Yêu cầu 5: Huấn luyện và kiểm tra mô hình Multinomial Naive Bayes (sử dụng phân phối đa thức)
######
# Code

mnb = MultinomialNB()
mnb.fit(X_ohe, y_enc)

pred_mnb = mnb.predict(new_ohe)
prob_mnb = mnb.predict_proba(new_ohe)

print('=== Multinomial Naive Bayes ===')
print(f'Kết quả dự đoán: {le.inverse_transform(pred_mnb)[0]}')
print(f'Xác suất [No, Yes]: {prob_mnb.round(4)}')
######

In [9]:
# Yêu cầu 6: Huấn luyện và kiểm tra mô hình mô hình Bernoulli Naive Bayes (sử dụng phân phối Bernoulli)
######
# Code

bnb = BernoulliNB()
bnb.fit(X_ohe, y_enc)

pred_bnb = bnb.predict(new_ohe)
prob_bnb = bnb.predict_proba(new_ohe)

print('=== Bernoulli Naive Bayes ===')
print(f'Kết quả dự đoán: {le.inverse_transform(pred_bnb)[0]}')
print(f'Xác suất [No, Yes]: {prob_bnb.round(4)}')
######

In [10]:
# Yêu cầu 7: Huấn luyện và kiểm tra mô hình Categorical Naive Bayes (Categorical distribution - mở rộng của phân phối Bernoulli)
######
# Code

# CategoricalNB yêu cầu dữ liệu dạng số nguyên không âm (Ordinal Encoding)
catnb = CategoricalNB()
catnb.fit(X_ordinal, y_enc)

pred_catnb = catnb.predict(new_ordinal)
prob_catnb = catnb.predict_proba(new_ordinal)

print('=== Categorical Naive Bayes ===')
print(f'Kết quả dự đoán: {le.inverse_transform(pred_catnb)[0]}')
print(f'Xác suất [No, Yes]: {prob_catnb.round(4)}')
######

In [11]:
# Yêu cầu 8: Huấn luyện và kiểm tra mô hình Complement Naive Bayes (điều chỉnh của phân phối đa thức)
######
# Code

cnb = ComplementNB()
cnb.fit(X_ohe, y_enc)

pred_cnb = cnb.predict(new_ohe)
prob_cnb = cnb.predict_proba(new_ohe)

print('=== Complement Naive Bayes ===')
print(f'Kết quả dự đoán: {le.inverse_transform(pred_cnb)[0]}')
print(f'Xác suất [No, Yes]: {prob_cnb.round(4)}')

# Tổng hợp kết quả
print('\n========== Tổng hợp kết quả trên mẫu mới ==========')
results = {
    'GaussianNB': le.inverse_transform(pred_gnb)[0],
    'MultinomialNB': le.inverse_transform(pred_mnb)[0],
    'BernoulliNB': le.inverse_transform(pred_bnb)[0],
    'CategoricalNB': le.inverse_transform(pred_catnb)[0],
    'ComplementNB': le.inverse_transform(pred_cnb)[0],
}
for model_name, prediction in results.items():
    print(f'  {model_name}: Defaulted Borrower = {prediction}')
######

## II. Phân loại Naive Bayes sử dụng thư viện sklearn trên dataset lớn (Dữ liệu: credit_data.csv)

- Bài toán: Dự đoán điểm tín dụng của khách hàng khi vay vốn sử dụng Naïve Bayes.
- Mục tiêu:
	- Xây dựng được mô hình Naïve Bayes sử dụng thư viện sklearn.
	- Ứng dụng và hiểu cách áp dụng mô hình Naïve Bayes vào giải quyết bài toán thực tế (ví dụ: dự đoán điểm tín dụng).
	- Sử dụng độ đo Accuracy để đánh giá chất lượng mô hình.
	- Thay đổi các phân bố xác suất (phân phối chuẩn, phân phối đa thức, phân phối Bernoulli) để chọn ra bộ phận lớp Naive Bayes phù hợp với dữ liệu.
- Dữ liệu:
	- Tập các đặc trưng của khách hàng và điểm tín dụng tương ứng trong một khoảng thời gian nhất định.
	- Tập các nhãn (Cột “Risk”): Gồm 2 loại nhãn “Good” và “Bad”. Trong đó “Good” biểu thị khách hàng có khả năng trả nợ đúng hạn, “Bad” biểu thị khách hàng có khả năng vỡ nợ.
- Loại bài toán: Phân loại
	- Input: n vector đã mã hóa của các khách hàng.
	- Output: nhãn y là một trong 2 nhãn trên.


### 1. Import các thư  viện cần thiết

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score

### 2. Load dữ liệu

In [13]:
# Yêu cầu 1: Đọc dữ liệu từ file csv, hiển thị 5 dòng đầu tiên trong dataset
######
# Code

df = pd.read_csv('credit_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()
######

### 3. Tiền xử lý dữ liệu

#### 3.1. Hiểu dữ liệu

In [14]:
# Yêu cầu 2: Hiển thị thông tin tổng quan của dataset (số dòng, số cột, tên các cột, kiểu dữ liệu)
# Viết nhận xét về kết quả thu được
######
# Code

print(f'Số dòng: {df.shape[0]}, Số cột: {df.shape[1]}')
print(f'\nTên các cột: {list(df.columns)}')
print()
df.info()

# Nhận xét:
# - Dataset gồm nhiều đặc trưng mô tả hồ sơ tín dụng của khách hàng.
# - Có cả cột kiểu số (int/float) và kiểu chuỗi (object) cần được mã hóa trước khi đưa vào mô hình.
# - Cột 'Risk' là nhãn phân loại (Good/Bad).
######

In [15]:
# Yêu cầu 3: Hiển thị thông tin các thống kê mô tả cơ bản của các đặc trưng
# Gợi ý: Sử dụng hàm describe()
######
# Code

print('Thống kê mô tả các đặc trưng số:')
df.describe()
######

#### 3.2. Kiểm tra missing values

In [16]:
# Yêu cầu 4: Thống kê tổng số giá trị bị thiếu trong dataset, liệt kê giá trị bị thiếu trong mỗi cột.
# Viết nhận xét cho kết quả thu được
######
# Code

missing_total = df.isnull().sum().sum()
missing_per_col = df.isnull().sum()

print(f'Tổng số giá trị bị thiếu trong dataset: {missing_total}')
print('\nSố giá trị bị thiếu theo từng cột:')
print(missing_per_col[missing_per_col > 0] if missing_total > 0 else 'Không có giá trị bị thiếu.')

# Nhận xét:
# - Nếu có missing values, cần xử lý trước khi huấn luyện mô hình.
# - Các cột số có thể điền bằng mean/median; cột phân loại có thể điền bằng mode.
######

#### 3.3. Xử lý missing values

In [17]:
# Yêu cầu 5: Lựa chọn và áp dụng phương pháp xử lý các giá trị bị thiếu
######
# Code

# Điền missing values:
# - Cột số: dùng median (ít bị ảnh hưởng bởi outlier)
# - Cột phân loại (object): dùng mode (giá trị phổ biến nhất)

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

print(f'Tổng missing values sau xử lý: {df.isnull().sum().sum()}')
######

#### 3.4. Mã hóa các đặc trưng rời rạc

In [18]:
# Yêu cầu 6: Chuyển đổi các giá trị rời rạc về giá trị số
# Gợi ý: Có thể áp dụng một trong các kỹ thuật: Label Encoding, One-Hot Encoding, Ordinal Encoding, Target Encoding
######
# Code

from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
label_encoders = {}

# Áp dụng Label Encoding cho tất cả cột kiểu object
for col in df_encoded.select_dtypes(include='object').columns:
    le_col = LabelEncoder()
    df_encoded[col] = le_col.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le_col
    print(f'Cột [{col}] -> classes: {list(le_col.classes_)}')

print('\nDataset sau mã hóa:')
print(df_encoded.head())
######

### 4. Chia dữ liệu thành 2 phần: Training và Testing

In [19]:
# Yêu cầu 7: Chia dữ liệu thành 80% dùng để train và 20% dùng để test.
######
# Code

X2 = df_encoded.drop(columns=['Risk'])
y2 = df_encoded['Risk']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

print(f'Tập train: {X2_train.shape[0]} mẫu')
print(f'Tập test : {X2_test.shape[0]} mẫu')
######

### 5. Training and Testing Naive Bayes model

In [20]:
# Yêu cầu 8: Chuẩn hóa toàn bộ dữ liệu về một phạm vi nhất định (Data Scaling)
# Gợi ý: Có thể sử dụng StandardScaler, MinMaxScaler, Robust Scaling
######
# Code

# StandardScaler: phù hợp với GaussianNB (giả định phân phối chuẩn)
std_scaler = StandardScaler()
X2_train_std = std_scaler.fit_transform(X2_train)
X2_test_std  = std_scaler.transform(X2_test)

# MinMaxScaler: phù hợp với MultinomialNB, BernoulliNB (yêu cầu giá trị không âm)
mm_scaler = MinMaxScaler()
X2_train_mm = mm_scaler.fit_transform(X2_train)
X2_test_mm  = mm_scaler.transform(X2_test)

print('Chuẩn hóa hoàn tất.')
print(f'X_train_std shape: {X2_train_std.shape}')
print(f'X_train_mm  shape: {X2_train_mm.shape}')
######

In [21]:
# Training & Testing
# Yêu cầu 9: Sử dụng thư viện sklearn để xây dựng một mô hình Naive Bayes và kiểm tra kết quả phân lớp dựa trên độ đo Accuracy.
######
# Code

gnb2 = GaussianNB()
gnb2.fit(X2_train_std, y2_train)
y2_pred_gnb = gnb2.predict(X2_test_std)
acc_gnb = accuracy_score(y2_test, y2_pred_gnb)

print('=== Gaussian Naive Bayes ===')
print(f'Accuracy: {acc_gnb:.4f} ({acc_gnb*100:.2f}%)')
######

### 6. Thay đổi phân bố xác suất để chọn được bộ phân lớp phù hợp với dữ liệu

In [22]:
# Gợi ý: Khảo sát các phân bố xác suất như phân phối chuẩn (Gauss), phân phối đa thức, phân phối Bernoulli và chọn ra bộ phân lớp tương ứng với phân phối cho accuracy cao nhất.
# Mở rộng (Tùy chọn): Điều chỉnh các tham số để cho kết quả tốt hơn (VD: giá trị làm mịn alpha trong phân phối đa thức, var_smoothing trong Gauss,...)
######
# Code

results_part2 = {}

# 1. Gaussian NB (StandardScaler)
for var_smooth in [1e-9, 1e-6, 1e-3]:
    m = GaussianNB(var_smoothing=var_smooth)
    m.fit(X2_train_std, y2_train)
    acc = accuracy_score(y2_test, m.predict(X2_test_std))
    results_part2[f'GaussianNB (var_smoothing={var_smooth})'] = acc

# 2. Multinomial NB (MinMaxScaler - giá trị không âm)
for alpha in [0.1, 0.5, 1.0, 2.0]:
    m = MultinomialNB(alpha=alpha)
    m.fit(X2_train_mm, y2_train)
    acc = accuracy_score(y2_test, m.predict(X2_test_mm))
    results_part2[f'MultinomialNB (alpha={alpha})'] = acc

# 3. Bernoulli NB (MinMaxScaler)
for alpha in [0.1, 0.5, 1.0]:
    m = BernoulliNB(alpha=alpha)
    m.fit(X2_train_mm, y2_train)
    acc = accuracy_score(y2_test, m.predict(X2_test_mm))
    results_part2[f'BernoulliNB (alpha={alpha})'] = acc

# Hiển thị kết quả
print('========== So sánh các mô hình Naive Bayes ==========')
best_model = max(results_part2, key=results_part2.get)
for model_name, acc in sorted(results_part2.items(), key=lambda x: -x[1]):
    marker = ' <-- TỐT NHẤT' if model_name == best_model else ''
    print(f'  {model_name}: Accuracy = {acc:.4f} ({acc*100:.2f}%){marker}')

print(f'\nKết luận: Mô hình tốt nhất là [{best_model}] với Accuracy = {results_part2[best_model]:.4f}')
######